# CCT 数据准备 — 流式取 1B tokens → Pack → 上传 Kaggle

**目的**: 从 HuggingFace 流式下载 50/30/20 混合数据 → Pack → 上传 Kaggle

**数据源**: `HuggingFaceFW/finepdfs_50BT-dclm_30BT-fineweb_edu_20BT-shuffled`
- 50% FinePDFs (学术 PDF) + 30% DCLM (web) + 20% FineWeb-Edu (教育)
- 已全局 shuffle (seed=42), 100B tokens, 我们只取 ~1B
- 来源: codelion "The 1 Billion Token Challenge" (2025)

**输出**: 上传到 `wukeneth/cct-pretrain-data`

## 0. 安装依赖 + Kaggle 认证

In [ ]:
!pip install -q torch transformers datasets accelerate kaggle

In [ ]:
# === Kaggle API 认证 ===
import os, json

KAGGLE_CREDS = {"username": "wukeneth", "key": "51c30f09c0d8ec8e5884a4c16b49a0a5"}

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_path = os.path.join(kaggle_dir, 'kaggle.json')
with open(kaggle_path, 'w') as f:
    json.dump(KAGGLE_CREDS, f)
os.chmod(kaggle_path, 0o600)
print('kaggle.json 已写入 %s' % kaggle_path)

## 1. 配置

In [ ]:
import os
import torch
from transformers import AutoTokenizer
from datasets import load_dataset
from torch.utils.data import IterableDataset

MODEL_NAME = "unsloth/Llama-3.2-1B"

# === Packing 参数 (与训练 notebook 对齐) ===
SEQ_LEN = 2048
BATCH_SIZE = 32
GRAD_ACCUM = 4
MAX_STEPS = 3800
EVAL_CHUNKS = 100

# 数据源: HuggingFaceFW 官方 50/30/20 混合 (已 shuffle)
DS_NAME = 'HuggingFaceFW/finepdfs_50BT-dclm_30BT-fineweb_edu_20BT-shuffled'

# 计算需要的总 chunk 数
total_chunks_needed = MAX_STEPS * GRAD_ACCUM * BATCH_SIZE
total_tokens_needed = total_chunks_needed * SEQ_LEN
print('Seq len: %d' % SEQ_LEN)
print('Batch size: %d, Grad accum: %d' % (BATCH_SIZE, GRAD_ACCUM))
print('Max steps: %d' % MAX_STEPS)
print('Total chunks needed: %d (%.1fM tokens ≈ %.2fB tokens)' % (
    total_chunks_needed, total_tokens_needed / 1e6, total_tokens_needed / 1e9))
print()
print('Data source: %s' % DS_NAME)

## 2. 流式下载 + Packing

In [ ]:
# === Tokenizer ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print('Vocab: %d, EOS: %d' % (tokenizer.vocab_size, tokenizer.eos_token_id))

# === Packing Dataset ===
class PackedDataset(IterableDataset):
    """流式 tokenize + pack: 拼接文档 (EOS 分隔) → 切 seq_len 块"""
    def __init__(self, hf_dataset, tokenizer, seq_len=2048):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.seq_len = seq_len

    def __iter__(self):
        buffer = []
        for example in self.dataset:
            text = example.get('text', '')
            if not text or not text.strip():
                continue
            ids = self.tokenizer(text, add_special_tokens=False)['input_ids']
            buffer.extend(ids)
            buffer.append(self.tokenizer.eos_token_id)
            while len(buffer) >= self.seq_len:
                chunk = buffer[:self.seq_len]
                buffer = buffer[self.seq_len:]
                t = torch.tensor(chunk, dtype=torch.long)
                yield {
                    'input_ids': t,
                    'attention_mask': torch.ones_like(t),
                    'labels': t.clone(),
                }

# === 流式数据 (单源, 已混合+shuffle) ===
print('Loading streaming dataset: %s' % DS_NAME)
ds_train = load_dataset(DS_NAME, split='train', streaming=True)
ds_eval = load_dataset(DS_NAME, split='train', streaming=True)

# Eval 跳过前 N 条避免与 train 重叠
# (数据集已 shuffle, 但为安全起见 skip 一些)
EVAL_SKIP = 500000
ds_eval = ds_eval.skip(EVAL_SKIP)

# Eval: 收集固定 chunks
print('Packing eval chunks (%d) (skip first %d docs)...' % (EVAL_CHUNKS, EVAL_SKIP))
eval_packed = PackedDataset(ds_eval, tokenizer, seq_len=SEQ_LEN)
eval_data = []
for item in eval_packed:
    eval_data.append(item)
    if len(eval_data) >= EVAL_CHUNKS:
        break
print('Eval: %d chunks collected' % len(eval_data))

## 3. 保存 Pre-packed Data

In [ ]:
# === 保存到本地 ===
import os, json, time

os.makedirs('train_packed', exist_ok=True)
os.makedirs('eval_packed', exist_ok=True)

# 保存 eval
torch.save(eval_data, 'eval_packed/eval_chunks.pt')
print('Eval saved: %d chunks' % len(eval_data))

# 保存 train: 按 optimizer step 打包
print('Packing and saving train data...')
train_packed = PackedDataset(ds_train, tokenizer, seq_len=SEQ_LEN)
train_iter = iter(train_packed)

file_idx = 0
total_saved = 0
chunks_per_file = BATCH_SIZE * GRAD_ACCUM
t0 = time.time()

while total_saved < total_chunks_needed:
    batch_data = []
    for _ in range(chunks_per_file):
        try:
            item = next(train_iter)
            batch_data.append(item)
        except StopIteration:
            break
    if not batch_data:
        break
    torch.save(batch_data, 'train_packed/step_%05d.pt' % file_idx)
    total_saved += len(batch_data)
    file_idx += 1
    if file_idx % 100 == 0:
        elapsed = time.time() - t0
        tokens_done = total_saved * SEQ_LEN
        speed = tokens_done / elapsed if elapsed > 0 else 0
        print('  [%d files] %d chunks (%.1fM tokens) | %.0f tok/s | %.1f min' % (
            file_idx, total_saved, tokens_done / 1e6, speed, elapsed / 60))

elapsed = time.time() - t0
print('Train saved: %d files, %d chunks (%.2fB tokens) in %.1f min' % (
    file_idx, total_saved, total_saved * SEQ_LEN / 1e9, elapsed / 60))

# 保存 metadata
metadata = {
    'seq_len': SEQ_LEN,
    'batch_size': BATCH_SIZE,
    'grad_accum': GRAD_ACCUM,
    'max_steps': MAX_STEPS,
    'eval_chunks': EVAL_CHUNKS,
    'total_train_chunks': total_saved,
    'total_train_tokens': total_saved * SEQ_LEN,
    'num_train_files': file_idx,
    'data_source': DS_NAME,
}
with open('metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('Metadata saved')

## 4. 打包上传到 Kaggle

In [ ]:
# === 打包为 tar.gz 以加快上传 ===
import subprocess, shutil

SLUG = 'cct-pretrain-data'
KAGGLE_USER = 'wukeneth'
upload_dir = '/content/upload_pretrain'

if os.path.exists(upload_dir):
    shutil.rmtree(upload_dir)
os.makedirs(upload_dir, exist_ok=True)

# dataset-metadata.json (Kaggle 要求)
meta = {
    'title': 'CCT Pretrain Data (50/30/20 packed)',
    'id': '%s/%s' % (KAGGLE_USER, SLUG),
    'licenses': [{'name': 'Apache 2.0'}],
}
with open(os.path.join(upload_dir, 'dataset-metadata.json'), 'w') as f:
    json.dump(meta, f, indent=2)

# 复制 packed 数据到上传目录
shutil.copytree('train_packed', os.path.join(upload_dir, 'train_packed'))
shutil.copytree('eval_packed', os.path.join(upload_dir, 'eval_packed'))
shutil.copy2('metadata.json', os.path.join(upload_dir, 'metadata.json'))

# 统计大小
total_size = 0
for dirpath, dirnames, filenames in os.walk(upload_dir):
    for f in filenames:
        total_size += os.path.getsize(os.path.join(dirpath, f))
print('Upload dir: %s (%.2f GB)' % (upload_dir, total_size / 1e9))

In [ ]:
# === 上传到 Kaggle ===
import subprocess

def kaggle_upload(upload_dir, slug, update=False):
    if update:
        cmd = ['kaggle', 'datasets', 'version', '-p', upload_dir,
               '-m', 'auto update', '--dir-mode', 'zip']
    else:
        cmd = ['kaggle', 'datasets', 'create', '-p', upload_dir, '--dir-mode', 'zip']
    print('[%s] %s' % (slug, ' '.join(cmd)))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        if 'already exists' in r.stderr.lower() or 'already exists' in r.stdout.lower():
            print('[%s] 数据集已存在, 改用 version 更新...' % slug)
            return kaggle_upload(upload_dir, slug, update=True)
        print('[%s] 错误: %s' % (slug, r.stderr))
        return False
    return True

print('=== 上传 %s ===' % SLUG)
kaggle_upload(upload_dir, SLUG)

## 5. 验证

In [ ]:
# === 验证上传 ===
print('=== Kaggle Datasets ===')
!kaggle datasets list -m --user wukeneth

print()
print('上传完成!')
print('在 Kaggle Notebook 中添加 Dataset:')
print('  wukeneth/cct-pretrain-data')
print()
print('数据格式:')
print('  train_packed/step_NNNNN.pt  — 每个含 %d chunks (1 optimizer step)' % (BATCH_SIZE * GRAD_ACCUM))
print('  eval_packed/eval_chunks.pt  — %d eval chunks' % EVAL_CHUNKS)
print('  metadata.json               — 配置信息')